# PA4 Task 2 — follow-up: what in mixed data blocks window / dilated learning (~35 min on 2 × T4)

The first diagnostic (D7) showed that window and dilated attention fail on mixed data even when only the reachable
(d ≤ 63) queries are supervised, while near data with the same number of supervised queries learns. Reachable queries
sit at the same positions and distances in both regimes; what differs is the input around them (~7.5 other queries in
their window on near data vs ~1.6 on mixed data, plus ~12.7 far query tokens per mixed sequence).

| GPU 0 | GPU 1 |
|---|---|
| near data with only 20.7 % of the queries left in the input; mixed data with the far query tokens removed (window and dilated, 3 seeds each) | the main window-mixed condition with 3 × the update budget (3 seeds) |

No checkpoints and no Qwen download are needed. Output: `results/diagnostics/followup/`, table D10, figure FD5, zipped
as `task2_followup.zip`. Settings: **GPU T4 × 2**, **Internet on**, **Save & Run All**.

In [ ]:
REPO_URL = "https://github.com/wasaymj/ATML.git"
BRANCH   = "main"

import os, subprocess, sys, shutil, glob, time
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 2"
LOGS = f"{WORK}/logs"
CACHE = f"{WORK}/mqar_cache"
NL = chr(10)
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{TASK}/results/diagnostics/part_b_diagnostic.json"), "commit and push the first diagnostic results"
assert "followup" in open(f"{TASK}/scripts/diagnose_task2.py").read(), "push the follow-up code first"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)

In [ ]:
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

def launch(name, gpu, arms):
    script = f"{LOGS}/{name}.sh"
    cmd = (f'{sys.executable} scripts/diagnose_task2.py --part b --steps followup --arms {arms} '
           f'--cache_dir "{CACHE}_{name}" --ckpt_dir "{WORK}/followup_ckpt" || echo "STEP FAILED"')
    open(script, "w").write(f'cd "{TASK}"' + NL + cmd + NL)
    print(f"[{name}] GPU {gpu}: {arms}", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=5):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep=NL + "   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

SHORT = "window_near_sparse_input,window_mixed_far_removed,dilated_near_sparse_input,dilated_mixed_far_removed"
LONG = "window_mixed_long"
import torch
if torch.cuda.device_count() >= 2:
    wait({"followup_short": launch("followup_short", 0, SHORT), "followup_long": launch("followup_long", 1, LONG)})
else:
    wait({"followup_all": launch("followup_all", 0, SHORT + "," + LONG)})
for f in glob.glob(f"{LOGS}/*.log"):
    print(f, "<-- contains a failed step" if "STEP FAILED" in open(f, errors="replace").read() else "ok")

In [ ]:
rc = subprocess.run([sys.executable, "scripts/compile_task2_report.py"], cwd=TASK).returncode
print("report exit code", rc)
shutil.copytree(LOGS, f"{TASK}/results/diagnostics/followup/logs", dirs_exist_ok=True)
out = f"{WORK}/task2_followup"
shutil.rmtree(out, ignore_errors=True)
shutil.copytree(f"{TASK}/results/diagnostics/followup", f"{out}/results/diagnostics/followup")
os.makedirs(f"{out}/results/tables", exist_ok=True); os.makedirs(f"{out}/results/figures", exist_ok=True)
for f in glob.glob(f"{TASK}/results/tables/D10_*"):
    shutil.copy(f, f"{out}/results/tables/")
for f in glob.glob(f"{TASK}/results/figures/FD5_*"):
    shutil.copy(f, f"{out}/results/figures/")
shutil.make_archive(f"{WORK}/task2_followup", "zip", out, "results")
print(sorted(os.listdir(WORK)))

In [ ]:
from IPython.display import Markdown, Image, display
for t in glob.glob(f"{TASK}/results/tables/D10_*.md"):
    display(Markdown(open(t, encoding="utf-8").read()))
for fig in glob.glob(f"{TASK}/results/figures/FD5_*.png"):
    display(Image(fig, width=900))